# **Importing Necessary Libraries**


In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


# **Explore Data🔎**


In [4]:
# Works both in the Replit workspace and after uploading the files to Kaggle/Jupyter.
DATA_PATHS = [
    '../data/attached_assets/coffee_shop_concat_1790089302722.csv',
    '../data/coffee_shop_concat_1790089302722.csv',
]
DATA_PATH = next((path for path in DATA_PATHS if os.path.exists(path)), DATA_PATHS[0])
data = pd.read_csv(DATA_PATH)


FileNotFoundError: [Errno 2] No such file or directory: '../data/attached_assets/coffee_shop_concat_1790089302722.csv'

In [ ]:
data.head(10)


In [ ]:
data.shape


In [ ]:
data.info()


In [ ]:
data.describe(include='all').T


In [ ]:
print(data.dtypes)
print('Columns:', data.columns.tolist())


In [ ]:
print('Duplicate rows:', data.duplicated().sum())
print('Duplicate Transaction IDs:', data['Transaction ID'].duplicated().sum())


# **Standardize Missing Values**


In [ ]:
# The actual CSV has 7 columns. It does not contain Age, Payment Method,
# Total Spent, Item, or Price Per Unit from the older notebook schema.
MISSING_TOKENS = {
    '', 'UNKNOWN', 'ERROR', 'NA', 'N/A', 'NULL', 'NONE', 'NAN', '-1'
}
PREFER_NOT_TO_SAY = 'PREFER NOT TO SAY'

for col in data.select_dtypes(include=['object', 'string']).columns:
    data[col] = data[col].astype('string').str.strip()
    # Explicitly preserve the requested meaning of "prefer not to say".
    data[col] = data[col].mask(data[col].str.upper().eq(PREFER_NOT_TO_SAY), 'Unknown')
    data[col] = data[col].mask(data[col].str.upper().isin(MISSING_TOKENS), pd.NA)

# Convert numeric fields; invalid strings become missing.
for col in ['Quantity', 'Price']:
    data[col] = pd.to_numeric(data[col], errors='coerce')
    data.loc[data[col] < 0, col] = np.nan

# Parse dates; invalid dates become missing.
data['Transaction Date'] = pd.to_datetime(
    data['Transaction Date'], errors='coerce', format='%Y-%m-%d'
)


# **Missingness Profile**


In [ ]:
missing_summary = pd.DataFrame({
    'Missing Count': data.isna().sum(),
    'Missing Rate (%)': (data.isna().mean() * 100).round(3),
    'Dtype': data.dtypes.astype(str)
})
missing_summary


In [ ]:
# Missingness by the observed Source variable.
missing_by_source = (
    data.groupby('Source', dropna=False)
        .apply(lambda group: group.isna().sum())
)
missing_rate_by_source = (
    data.groupby('Source', dropna=False)
        .apply(lambda group: (group.isna().mean() * 100).round(3))
)
print('Missing counts by Source:')
display(missing_by_source)
print('Missing rates (%) by Source:')
display(missing_rate_by_source)


In [ ]:
# Missingness patterns: useful for checking whether fields are missing together.
missing_columns = ['Transaction Date', 'Product Name', 'Quantity', 'Price', 'Store Location']
missing_pattern = (
    data[missing_columns].isna()
        .astype(int)
        .astype(str)
        .agg(''.join, axis=1)
        .value_counts()
        .rename_axis('Pattern')
        .reset_index(name='Rows')
)
missing_pattern.head(15)


# **Classify Missingness: MCAR / MAR / MNAR**

- **MCAR**: missingness is independent of both observed and unobserved values.
- **MAR**: missingness can be explained by observed variables, here especially `Source`.
- **MNAR**: missingness depends on the unobserved value itself or another unobserved cause.

For this dataset, the working classification is **MAR (observed)** for the columns with missing values: the missingness is concentrated in `Cafe Sales`, while `Coffee Shop Sales` has no missing values in the same fields. This is a source/collection-structure signal, not evidence of MCAR. MNAR cannot be proven from the observed table alone.


In [ ]:
mechanism_table = pd.DataFrame([
    ['Transaction ID', 'Not applicable', 'No missing values; this is the record key.'],
    ['Transaction Date', 'MAR (observed)', 'Missing rate differs by Source: 4.6% in Cafe Sales versus 0% in Coffee Shop Sales.'],
    ['Product Name', 'MAR (observed)', 'Missing rate differs by Source: 9.69% versus 0%.'],
    ['Quantity', 'MAR (observed)', 'Missing rate differs by Source: 4.79% versus 0%.'],
    ['Price', 'MAR (observed)', 'Missing rate differs by Source: 5.33% versus 0%.'],
    ['Store Location', 'MAR / structural missing', 'Missing or sentinel rate is 39.61% in Cafe Sales versus 0%.'],
    ['Source', 'Not applicable', 'No missing values; used as an observed grouping variable.'],
], columns=['Column', 'Mechanism', 'Reason'])
mechanism_table


# **Handle Missing Values**


In [ ]:
# Numeric distribution check: choose median when the distribution is skewed.
numeric_summary = data[['Quantity', 'Price']].agg(['count', 'mean', 'median', 'std', 'min', 'max']).T
numeric_summary['Mean/Median Ratio'] = (numeric_summary['mean'] / numeric_summary['median']).round(3)
numeric_summary


In [ ]:
# Target handling.
# No target column was supplied in the actual CSV. Transaction ID is a key,
# not a prediction target. Transaction Date is required for time-based analysis.
TARGET_COLUMNS = []


## **Clean required fields**

Rows with a missing/invalid `Transaction ID` or `Transaction Date` are removed with `dropna(...).reset_index(drop=True)`. Forward-fill is intentionally not used because the source file is not ordered chronologically and `Transaction ID` is not a time sequence.


In [ ]:
rows_before = len(data)
data = data.dropna(subset=['Transaction ID', 'Transaction Date']).reset_index(drop=True)
rows_dropped = rows_before - len(data)
print('Rows before:', rows_before)
print('Rows dropped:', rows_dropped)
print('Rows after:', len(data))


## **Fill numeric columns: median by Source**


In [ ]:
numeric_imputed = {}
for col in ['Quantity', 'Price']:
    before = data[col].isna().sum()
    data[col] = data.groupby('Source')[col].transform(
        lambda s: s.fillna(s.median())
    )
    numeric_imputed[col] = before

print('Numeric values imputed:', numeric_imputed)


## **Fill categorical columns: mode by Source**


In [ ]:
def fill_with_source_mode(series):
    # Do not overwrite an explicit "Unknown" value created from
    # "prefer not to say"; only fill actual missing values.
    observed = series.dropna()
    observed = observed[observed != 'Unknown']
    if observed.empty:
        return series.fillna('Unknown')
    return series.fillna(observed.mode().iloc[0])

categorical_imputed = {}
for col in ['Product Name', 'Store Location']:
    before = data[col].isna().sum()
    data[col] = data.groupby('Source')[col].transform(fill_with_source_mode)
    categorical_imputed[col] = before

print('Categorical values imputed:', categorical_imputed)


## **Data after processing**


In [ ]:
data.head(20)


In [ ]:
data.info()


In [ ]:
data.isna().sum()


# **Detect Outliers**


In [ ]:
for col in ['Quantity', 'Price']:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((data[col] < lower) | (data[col] > upper)).sum()
    print(f'{col}: {outliers} outliers; bounds=({lower:.3f}, {upper:.3f})')


In [ ]:
for col in ['Quantity', 'Price']:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=data[col])
    plt.title(f'Boxplot of {col}')
    plt.show()


# **Validate the Dataset**


In [ ]:
print('Missing values after cleaning:')
print(data.isna().sum())
print('\nData types:')
print(data.dtypes)
print('\nUnique values in categorical columns:')
for col in ['Product Name', 'Store Location', 'Source']:
    print(f'\n{col}:')
    print(data[col].unique())


In [ ]:
validation = {
    'rows': len(data),
    'columns': len(data.columns),
    'duplicate_rows': int(data.duplicated().sum()),
    'duplicate_transaction_ids': int(data['Transaction ID'].duplicated().sum()),
    'missing_cells': int(data.isna().sum().sum()),
    'invalid_dates': int(data['Transaction Date'].isna().sum()),
    'negative_quantity': int((data['Quantity'] < 0).sum()),
    'negative_price': int((data['Price'] < 0).sum()),
}
validation


# **Save Cleaned Dataset**


In [ ]:
OUTPUT_PATH = 'Cleaned_Final_DataSet.csv'
data.to_csv(OUTPUT_PATH, index=False)
print(f'Saved cleaned dataset to: {OUTPUT_PATH}')


## **Data Quality Note**

`Store Location` currently mixes geographic store locations (`Hell's Kitchen`, `Astoria`, `Lower Manhattan`) with service channels (`Takeaway`, `In-store`). Before analyzing location, confirm the business meaning and consider splitting the column into `Store Location` and `Order Channel`.
